# Scenario Uncertainty in Climate Projections
<br>

**Scenario uncertainty** is the second of three sources of uncertainty in climate
projections, alongside model uncertainty and internal variability. It arises
because the future trajectory of greenhouse gas (GHG) emissions depends on
decisions — societal, economic, and political — that have not yet been made and
cannot be predicted. Climate models are deterministic given a forcing: run the
same model with the same GHG concentration pathway and you get the same climate.
The uncertainty is in which pathway the world will follow.

The scientific community represents this uncertainty through **Shared
Socioeconomic Pathways (SSPs)** — a set of plausible future worlds defined by
narratives about population growth, economic development, energy technology, and
land use. Each SSP produces a distinct GHG concentration trajectory, and
therefore a distinct climate projection:

| Pathway | Radiative forcing by 2100 | Rough narrative |
|---|---|---|
| SSP2-4.5 | 4.5 W/m² | Middle of the road — moderate mitigation |
| SSP3-7.0 | 7.0 W/m² | Regional rivalry — high emissions, weak cooperation |
| SSP5-8.5 | 8.5 W/m² | Fossil-fuelled development — highest emissions |

**Two key properties of scenario uncertainty distinguish it from the other
two sources:**
1. It is **not reducible by adding more models or ensemble members** — it
   reflects genuine ignorance of the future, not a sampling problem.
2. It **grows with time** — in the near term (< ~2040), all three pathways
   produce nearly identical projections because atmospheric GHG concentrations
   have not yet diverged enough to matter. By end-of-century, scenario choice
   dominates total uncertainty for temperature.

**This notebook covers:**
1. **Visualizing** scenario spread — how and when the three pathways diverge
   (Steps 1–2)
2. **Quantifying** scenario vs. model uncertainty over time — and finding the
   crossover point (Step 3)
3. **Mapping** where in California scenario choice matters most (Step 4)
4. **Decision guidance** — a practical checklist for choosing which SSPs to
   include in an analysis (Step 5)

> **For deeper scientific treatment**, see the
> [IPCC AR6 Synthesis Report](https://www.ipcc.ch/report/ar6/syr/),
> [Tebaldi et al. (2024) — npj Climate Action](https://www.nature.com/articles/s44168-023-00082-1),
> [IPCC uncertainty guidance note](https://www.ipcc.ch/site/assets/uploads/2018/05/uncertainty-guidance-note.pdf),
> and [ClimateWest: Uncertainty 101](https://climatewest.ca/2022/09/27/uncertainty-101-understanding-climate-models/).

> **Note:** This notebook is part of a series. For inter-model spread see
> `model_uncertainty.ipynb`; for within-model internal variability see
> `internal_variability_guidance.ipynb`; for the full three-way decomposition
> see `uncertainty_decomposition.ipynb`.

**Runtime:** Approximately **10–15 minutes** with default settings.

## Step 0: Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import warnings

# ── Scientific stack ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr

# ── Plotting: cartopy (static, publication-style maps) ───────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ── Plotting: plotly (interactive, HTML-exportable) ───────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── climakitae ─────────────────────────────────────────────────────────────
import climakitae as ck
from climakitae.util.colormap import read_ae_colormap
from climakitae.new_core.user_interface import ClimateData
from climakitae.core.data_interface import DataParameters
from climakitae.explore.uncertainty import GWL_1981_2010_FILE, read_csv_file

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = "retina"

# ── Mode bar: no Plotly logo, no zoom/pan tools — download PNG only ────────
config = dict(
    displaylogo=False,
    scrollZoom=False,
    modeBarButtonsToRemove=[
        "zoom2d",
        "pan2d",
        "select2d",
        "lasso2d",
        "zoomIn2d",
        "zoomOut2d",
        "autoScale2d",
        "resetScale2d",
    ],
    # keep only "toImage" (download as png) visible on the mode bar
)

## Step 1: Retrieve data

We retrieve **monthly near-surface air temperature** (`tas`) from the eight
Cal-Adapt WRF downscaled models, for all three emissions scenarios and the
historical baseline. Temperature is used here because it has the cleanest
scenario signal — for precipitation, internal variability often dominates (see
`internal_variability_guidance.ipynb`).

We also retrieve **monthly precipitation** (`prec`) for comparison in Step 4,
where spatial maps will illustrate how differently the two variables respond to
scenario choice.

### Step 1a: Historical baseline (1981–2010)

The historical baseline is scenario-independent — all three pathways share the
same observed forcing history. We retrieve it once and reuse it across all
three scenario comparisons.

In [ ]:
# # Retrieve historical monthly max temperature from LOCA2
# cd.reset()
# hist_ds = (
#     cd
#     .catalog("cadcat")
#     .activity_id("LOCA2")
#     .institution_id("UCSD")
#     .table_id("mon")
#     .grid_label("d03")
#     .variable("tasmax")
#     .experiment_id("historical")
#     .get()
# )
# hist_ds = hist_ds.unify_chunks()

# # 1981–2010 baseline
# hist_ds = hist_ds.sel(time=slice("1981", "2010"))
# hist_ds

In [ ]:
# # Retrieve historical monthly temperature from all 8 Cal-Adapt WRF models
# cd_hist = ClimateData()
# cd_hist.catalog("cadcat")
# cd_hist.variable("t2")            # near-surface air temperature
# cd_hist.table_id("mon")
# cd_hist.activity_id("WRF")
# cd_hist.institution_id("UCLA")
# cd_hist.experiment_id(["historical"])
# cd_hist.grid_label("d02")
# cd_hist.processes({"filter_unadjusted_models": "no"})

# hist_tas_ds = cd_hist.get()
# hist_tas_ds = ck.load(hist_tas_ds.unify_chunks(), progress_bar=True)

# # 1981–2010 baseline
# hist_tas = hist_tas_ds.sel(time=slice("1981", "2010"))
# print(f"Historical temperature loaded: {hist_tas.sizes}")

### Step 1b: Future projections — all three SSPs

We retrieve the three scenarios in separate calls, each using the `ClimateData`
interface with the corresponding `experiment_id`. All other parameters are
identical, so the resulting datasets are directly comparable.

The future period runs from 2015 to 2100, covering the full divergence of the
three pathways. We concatenate along a `scenario` dimension for downstream
analysis.

In [ ]:
# ── Shared LOCA2 retrieval helper ─────────────────────────────────────────
def retrieve_loca2(scenario: str, var: str = "tasmax"):
    """
    Retrieve monthly max temperature from LOCA2 for one scenario.

    scenario : one of 'historical', 'ssp245', 'ssp370', 'ssp585'
    """
    climate_dict = {
        "catalog": "cadcat",
        "activity_id": "LOCA2",
        "institution_id": "UCSD",
        "experiment_id": scenario,
        "table_id": "mon", 
        "grid_label": "d03", 
        "variable_id": var, 
        "processes":{"filter_unadjusted_models":"no"}
    }
    
    ds = ck.ClimateData(verbosity=-2).load_query(climate_dict).get()  
    ds = ds.assign_coords(sim=[s.split(f"_{scenario}")[0] for s in ds.sim.values])
    
    return ds.unify_chunks()

In [ ]:
# Retrieve historical
print("Retrieving historical...")
hist_ds = retrieve_loca2(scenario="historical", var="tasmax")

# Retrieve each scenario — these are the three main lines in the plume chart
print("Retrieving SSP2-4.5...")
ssp245_ds = retrieve_loca2(scenario="ssp245", var="tasmax")

print("Retrieving SSP3-7.0...")
ssp370_ds = retrieve_loca2("ssp370")

print("Retrieving SSP5-8.5...")
ssp585_ds = retrieve_loca2("ssp585")

print("All scenarios loaded.")

In [ ]:
def collapse_ensemble_members(ds, sim_dim=None):
    """
    Average ensemble members within each GCM so every GCM contributes
    equally to the multi-model mean, regardless of member count.

    Automatically detects the simulation dimension name ('sim' or 'simulation').
    Groups by GCM prefix (everything before '_r') and averages within each group.
    """
    # Auto-detect simulation dimension
    if sim_dim is None:
        for candidate in ("sim", "simulation"):
            if candidate in ds.dims:
                sim_dim = candidate
                break
    if sim_dim is None:
        raise ValueError(f"No simulation dimension found. Available dims: {list(ds.dims)}")

    sims    = ds[sim_dim].values
    gcm_ids = [s.split("_r")[0] for s in sims]

    if len(set(gcm_ids)) == len(sims):
        # Already one member per GCM — rename dim to 'gcm' for consistency
        return ds.rename({sim_dim: "gcm"})

    ds = ds.assign_coords(gcm=(sim_dim, gcm_ids))
    # ds = ds.groupby("gcm").mean(sim_dim)
    
    return ds.groupby("gcm").mean(sim_dim).rename({"gcm":"sim"})

In [ ]:
hist_ds = collapse_ensemble_members(hist_ds, sim_dim="sim")
ssp245_ds = collapse_ensemble_members(ssp245_ds, sim_dim="sim")
ssp370_ds = collapse_ensemble_members(ssp370_ds, sim_dim="sim")
ssp585_ds = collapse_ensemble_members(ssp585_ds, sim_dim="sim")

In [ ]:
print(f"Historical temperature loaded: {hist_ds.sizes}")
print(f"ssp245 temperature loaded: {ssp245_ds.sizes}")
print(f"ssp370 temperature loaded: {ssp370_ds.sizes}")
print(f"ssp temperature loaded: {ssp585_ds.sizes}")

In [ ]:
# Slice to the history period
hist_ds = hist_ds.sel(time=slice("1950", "2016"))
print(f"Historical temperature loaded: {hist_ds.sizes}")

# Slice to the future period and align simulations across scenarios
future_slice = slice("2014", "2100")

ssp245_ds = ssp245_ds.sel(time=future_slice)
ssp370_ds = ssp370_ds.sel(time=future_slice)
ssp585_ds = ssp585_ds.sel(time=future_slice)

# Keep only simulations that exist in all three scenarios
common_sims = (
    set(ssp245_ds.sim.values)
    & set(ssp370_ds.sim.values)
    & set(ssp585_ds.sim.values)
    & set(hist_ds.sim.values)
)
common_sims = np.array(sorted(common_sims))
print(f"Common simulations across all scenarios: {len(common_sims)}")

ssp245_future = ssp245_ds.sel(sim=common_sims)
ssp370_future = ssp370_ds.sel(sim=ssp370_ds.sim.isin(common_sims))
ssp585_future = ssp585_ds.sel(sim=ssp585_ds.sim.isin(common_sims))
hist_ds       = hist_ds.sel(sim=hist_ds.sim.isin(common_sims))

In [ ]:
print(f"Historical temperature loaded: {hist_ds.sizes}")
print(f"ssp245 temperature loaded: {ssp245_future.sizes}")
print(f"ssp370 temperature loaded: {ssp370_future.sizes}")
print(f"ssp585 temperature loaded: {ssp585_future.sizes}")

In [ ]:
hist_ds = ck.load(hist_ds.isel(lat=slice(None,None,2), lon=slice(None,None,2)), progress_bar=True)
ssp245_future = ck.load(ssp245_future.isel(lat=slice(None,None,2), lon=slice(None,None,2)), progress_bar=True)
ssp370_future = ck.load(ssp370_future.isel(lat=slice(None,None,2), lon=slice(None,None,2)), progress_bar=True)
ssp585_future = ck.load(ssp585_future.isel(lat=slice(None,None,2), lon=slice(None,None,2)), progress_bar=True)

### Step 1c: Compute anomalies and area averages

We convert to temperature anomalies relative to the 1981–2010 historical mean
per simulation — this removes inter-model mean biases, so the three scenario
lines start near zero and diverge upward. We then area-average over California
to produce timeseries for plotting.

In [ ]:
# Historical mean per simulation (spatial field, for anomaly calculation)
hist_mean_per_sim = hist_ds["tasmax"].mean(dim="time")   # shape: (sim, y, x)

def compute_annual_anom_ts(future_ds, hist_mean):
    """
    Subtract per-simulation historical mean, take annual average,
    then area-average over California.

    Returns a DataArray of shape (sim, year) representing
    California-mean annual temperature anomaly.
    """
    # Anomaly relative to 1981–2010 baseline per simulation
    anom = future_ds["tasmax"] - hist_mean   # broadcasts over time

    # Annual average (resample to yearly)
    anom_ann = anom.resample(time="1YE").mean()

    # Area average — simple mean over y/x for WRF grid
    # (a proper cos-latitude weighting can replace this if needed)
    anom_ca = anom_ann.mean(dim=["lat", "lon"])

    return anom_ca.compute()


print("Computing anomaly timeseries...")
ts_245 = compute_annual_anom_ts(ssp245_future, hist_mean_per_sim)
ts_370 = compute_annual_anom_ts(ssp370_future, hist_mean_per_sim)
ts_585 = compute_annual_anom_ts(ssp585_future, hist_mean_per_sim)

# Historical reference timeseries (anomaly ≈ 0 by construction)
hist_anom_ann = (
    (hist_ds["tasmax"] - hist_mean_per_sim)
    .resample(time="1YE")
    .mean()
    .mean(dim=["lat", "lon"])
    .compute()
)

print("Done.")

## Step 2: Visualize scenario spread over time

### Step 2a: Plume chart — three scenarios through 2100

Each scenario's **multi-model mean (MMM)** is shown as a bold line; the
**shaded envelope** spans the full range across the 8 WRF models within that
scenario. The envelope shows within-scenario model uncertainty; the gap between
the bold lines shows between-scenario (scenario) uncertainty.

The key visual finding: the three bold lines overlap almost completely before
roughly 2035–2040 and then diverge steeply, with SSP5-8.5 pulling away from
SSP2-4.5 by more than the width of either model envelope by end-of-century.

In [ ]:
np.unique(ssp585_future.time.dt.year.values)

In [ ]:
# Get the last historical year value for each GCM
last_hist_year = hist_anom_ann.isel(time=-1)   # shape: (gcm,)
last_hist_time = hist_anom_ann.time.isel(time=-1)

# Prepend it to each future timeseries before plotting
def prepend_hist_endpoint(ts_future, last_hist):
    """Add the last historical point as the first point of the future series."""
    bridge = last_hist.expand_dims(time=[last_hist_time.values])
    return xr.concat([bridge, ts_future], dim="time")

ts_245 = prepend_hist_endpoint(ts_245, last_hist_year)
ts_370 = prepend_hist_endpoint(ts_370, last_hist_year)
ts_585 = prepend_hist_endpoint(ts_585, last_hist_year)

In [ ]:
# ── Plotly interactive version (HTML-exportable) ─────────────────────────
fig_plotly = go.Figure()

# Historical envelope
hist_years = hist_anom_ann.time.dt.year.values.tolist()
fig_plotly.add_trace(go.Scatter(
    x=hist_years + hist_years[::-1],
    y=hist_anom_ann.max(dim="sim").values.tolist()
      + hist_anom_ann.min(dim="sim").values[::-1].tolist(),
    fill="toself", fillcolor="rgba(136,136,136,0.15)",
    line=dict(width=1, color="rgba(136,136,136,0.25)"), showlegend=False, hoverinfo="skip",
))
fig_plotly.add_trace(go.Scatter(
    x=hist_years, y=hist_anom_ann.mean(dim="sim").values.tolist(),
    line=dict(color="#444444", width=1.8, dash="solid"),
    name="Historical multi-model mean",
    hovertemplate="Historical multi-model mean<br>Year: %{x}<br>Anomaly: %{y:.2f}°C<extra></extra>",
))

for label, ts in scenarios.items():
    color  = SCENARIO_COLORS[label]
    years  = ts.time.dt.year.values.tolist()
    mmm    = ts.mean(dim="sim").values.tolist()
    lo     = ts.min(dim="sim").values.tolist()
    hi     = ts.max(dim="sim").values.tolist()

    # Model spread envelope
    r, g, b = tuple(int(color.lstrip("#")[i:i+2], 16) for i in (0, 2, 4))
    fig_plotly.add_trace(go.Scatter(
        x=years + years[::-1],
        y=hi + lo[::-1],
        fill="toself", fillcolor=f"rgba({r},{g},{b},0.15)",
        line=dict(width=1, color=f"rgba({r},{g},{b},0.25)"), showlegend=False, hoverinfo="skip",
    ))
    # MMM line
    fig_plotly.add_trace(go.Scatter(
        x=years, y=mmm,
        line=dict(color=color, width=2.5),
        name=f"{label} multi-model mean",
        hovertemplate=f"{label}<br>Year: %{{x}}<br>multi-model mean: %{{y:.2f}}°C<extra></extra>",
    ))

fig_plotly.add_vline(x=2040, line_dash="dash", line_color="#999", line_width=1)
fig_plotly.add_hline(y=0, line_dash="dot", line_color="#bbb", line_width=0.8)

fig_plotly.update_layout(
    title=dict(
        text="California annual temperature anomaly — three emissions scenarios"
             "<br><sup>Shaded envelopes = model spread within each scenario</sup>",
        font=dict(size=14),
    ),
    xaxis_title="Year",
    yaxis_title="Temperature anomaly (°C) relative to 1981–2010",
    plot_bgcolor="white",
    width=900, height=500,
    legend=dict(x=0.02, y=0.97, xanchor="left", yanchor="top", font=dict(size=11)),
)
fig_plotly.show(config=config)
fig_plotly.write_html(
    "../uncertainty/figures/html/scenario_plume.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig_plotly.write_image("../uncertainty/figures/static/scenario_plume_plotly.png", scale=2)

**Reading the chart:** The three bold lines represent the multi-model mean
projection under each scenario. The shaded envelopes around each line show how
much the 8 WRF models disagree *within* that scenario — that is the model
uncertainty contribution. The gap *between* the three bold lines is the
scenario uncertainty contribution.

Before roughly 2040, the three envelopes largely overlap — scenario choice
makes little practical difference for planning at that horizon. After 2040 the
lines separate clearly, and by 2071–2100 the difference between SSP5-8.5 and
SSP2-4.5 is typically larger than the model spread within either scenario.

### Step 2b: Scenario spread timeseries

The spread between the highest and lowest scenario MMMs, plotted as a single
line, makes the growth of scenario uncertainty explicit. This is the quantity
most relevant for decision-makers: *by how much does my planning outcome change
depending on which emissions pathway the world follows?*

In [ ]:
# ── Compute spread (same as matplotlib version) ───────────────────────────
spread_years  = ts_245.time.dt.year.values
spread_ts     = ts_585.mean(dim="sim").values - ts_245.mean(dim="sim").values
spread_smooth = pd.Series(spread_ts, index=spread_years).rolling(10, center=True, min_periods=5).mean()
eoc_spread    = float(spread_smooth.loc[2090:2100].mean())

# ── Figure ────────────────────────────────────────────────────────────────
fig = go.Figure()

# Filled area — annual spread
fig.add_trace(go.Scatter(
    x=np.concatenate([spread_years, spread_years[::-1]]),
    y=np.concatenate([spread_ts, np.zeros(len(spread_years))]),
    fill="toself",
    fillcolor="rgba(192,102,74,0.30)",
    line=dict(width=0),
    name="Annual spread",
    hoverinfo="skip",
))

# 10-year rolling mean line
fig.add_trace(go.Scatter(
    x=spread_smooth.index,
    y=spread_smooth.values,
    line=dict(color="#C0664A", width=2.2),
    name="10-yr rolling mean",
    hovertemplate="Year: %{x}<br>Spread: %{y:.2f}°C<extra></extra>",
))

# Zero reference line
fig.add_hline(y=0, line=dict(color="#bbbbbb", width=0.8, dash="dot"))

# Divergence vertical line
fig.add_vline(
    x=2040,
    line=dict(color="#999999", width=1, dash="dash"),
    annotation_text="Divergence after 2040",
    annotation_position="bottom right",
    annotation_font=dict(size=12, color="#000000"),
)

# End-of-century annotation with arrow
fig.add_annotation(
    x=2095, y=eoc_spread,
    ax=2060, ay=eoc_spread * 0.55,
    xref="x", yref="y",
    axref="x", ayref="y",
    text=f"End-of-century spread ≈ {eoc_spread:.1f}°C",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#000000",
    arrowwidth=1.5,
    font=dict(size=12, color="#000000"),
    align="left",
)

fig.update_layout(
    title=dict(
        text="Growth of scenario uncertainty over time — California temperature",
        font=dict(size=14),
    ),
    xaxis=dict(
        title="Year",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    yaxis=dict(
        title="SSP5-8.5 MMM − SSP2-4.5 MMM (°C)",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
        rangemode="tozero",
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        x=0.02, y=0.97,
        xanchor="left", yanchor="top",
        font=dict(size=12),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#dddddd",
        borderwidth=0.5,
    ),
    width=850, height=500,
    margin=dict(l=60, r=30, t=70, b=60),
)

fig.show(config=config)
fig.write_html(
    "../uncertainty/figures/html/scenario_spread_ts.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("../uncertainty/figures/static/scenario_spread_ts.png", scale=2)

The spread is near zero through 2035, rises gradually through mid-century, and
accelerates sharply after 2060. The end-of-century value — roughly the printed
annotation above — represents the full range of plausible California warming
under current uncertainty about future emissions. This number cannot be reduced
by running more models; it can only be reduced if global emissions policy
narrows the range of plausible futures.

### Step 2c: Period-averaged comparison — three planning horizons

The plume chart shows trends; this bar chart makes the comparison between
planning horizons concrete and numerical. For each of three standard planning
windows, we show the scenario MMMs side by side with the model spread overlaid
as error bars.

In [ ]:
HORIZONS = {
    "Near-term<br>(2020–2040)":      slice("2020", "2040"),
    "Mid-century<br>(2041–2070)":    slice("2041", "2070"),
    "End-of-century<br>(2071–2100)": slice("2071", "2100"),
}

# Compute period means and model spread per scenario and horizon
results = {}
for horizon_label, tslice in HORIZONS.items():
    results[horizon_label] = {}
    for scen_label, ts in scenarios.items():
        period = ts.sel(time=tslice).mean(dim="time")
        results[horizon_label][scen_label] = {
            "mmm": float(period.mean(dim="sim")),
            "lo":  float(period.min(dim="sim")),
            "hi":  float(period.max(dim="sim")),
        }

horizons  = list(HORIZONS.keys())
scen_list = ["SSP2-4.5", "SSP3-7.0", "SSP5-8.5"]

fig = go.Figure()

for scen_label in scen_list:
    color = SCENARIO_COLORS[scen_label]
    mmms    = [results[h][scen_label]["mmm"] for h in horizons]
    errs_lo = [results[h][scen_label]["mmm"] - results[h][scen_label]["lo"] for h in horizons]
    errs_hi = [results[h][scen_label]["hi"] - results[h][scen_label]["mmm"] for h in horizons]

    fig.add_trace(go.Bar(
        name=scen_label,
        x=horizons,
        y=mmms,
        marker=dict(color=color, opacity=0.85, line=dict(width=0)),
        error_y=dict(
            type="data",
            symmetric=False,
            array=errs_hi,
            arrayminus=errs_lo,
            color="#1a1a1a",
            thickness=1.5,
            width=6,
        ),
        hovertemplate=(
            f"<b>{scen_label}</b><br>"
            "%{x}<br>"
            "MMM: %{y:.2f}°C<br>"
            "Range: %{customdata[0]:.2f}–%{customdata[1]:.2f}°C"
            "<extra></extra>"
        ),
        customdata=[
            [results[h][scen_label]["lo"], results[h][scen_label]["hi"]]
            for h in horizons
        ],
    ))

fig.update_layout(
    barmode="group",
    bargap=0.20,
    bargroupgap=0.06,
    title=dict(
        text=(
            "Period-mean California temperature anomaly by scenario<br>"
            "<sup>Error bars = full model spread (min–max across GCMs)</sup>"
        ),
        font=dict(size=14),
    ),
    xaxis=dict(
        title=None,
        showgrid=False,
        showline=False,
    ),
    yaxis=dict(
        title="Temperature anomaly (°C)<br>relative to 1981–2010",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
        rangemode="tozero",
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        x=0.02, y=0.97,
        xanchor="left", yanchor="top",
        font=dict(size=11),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#dddddd",
        borderwidth=0.5,
    ),
    width=850, height=540,
    margin=dict(l=70, r=30, t=90, b=60),
)

fig.show(config=config)
fig.write_html(
    "../uncertainty/figures/html/scenario_period_bars.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("../uncertainty/figures/static/scenario_period_bars.png", scale=2)

**Reading the chart:** The bars are the multi-model means; the error bars span
the full range across the 8 WRF models within each scenario. In the near-term
panel, the three bars are nearly identical — scenario choice is irrelevant at
that horizon. In the end-of-century panel, the bars are well-separated and
each error bar (model spread) is smaller than the gap between adjacent bars
(scenario spread) — scenario choice is the dominant source of uncertainty.

In [ ]:
HORIZONS = {
    "Near-term<br>(2020–2040)":      slice("2020", "2040"),
    "Mid-century<br>(2041–2070)":    slice("2041", "2070"),
    "End-of-century<br>(2071–2100)": slice("2071", "2100"),
}

results = {}
for horizon_label, tslice in HORIZONS.items():
    results[horizon_label] = {}
    for scen_label, ts in scenarios.items():
        period = ts.sel(time=tslice).mean(dim="time")
        results[horizon_label][scen_label] = {
            # # Min-max approach
            # "mmm": float(period.mean(dim="sim")),
            # "lo":  float(period.min(dim="sim")),
            # "hi":  float(period.max(dim="sim")),
            # ±1 standard
            "mmm": float(period.mean(dim="sim")),
            "lo":  float(period.mean(dim="sim") - period.std(dim="sim")),
            "hi":  float(period.mean(dim="sim") + period.std(dim="sim")),
            # # IQR approach
            # "mmm": float(period.mean(dim="sim")),
            # "lo": float(period.quantile(0.25, dim="sim")),
            # "hi": float(period.quantile(0.75, dim="sim")),
        }

horizons  = list(HORIZONS.keys())
scen_list = ["SSP2-4.5", "SSP3-7.0", "SSP5-8.5"]

# vertical offsets so the three scenario rows within a horizon don't overlap
n_scen = len(scen_list)
offsets = np.linspace(0.22, -0.22, n_scen)  # top to bottom within each horizon group

fig = go.Figure()

for i, scen_label in enumerate(scen_list):
    color = SCENARIO_COLORS[scen_label]
    y_positions = [h_idx + offsets[i] for h_idx in range(len(horizons))]

    mmms = [results[h][scen_label]["mmm"] for h in horizons]
    los  = [results[h][scen_label]["lo"] for h in horizons]
    his  = [results[h][scen_label]["hi"] for h in horizons]

    # range line (model spread) — drawn first so dots sit on top
    for y, lo, hi in zip(y_positions, los, his):
        fig.add_trace(go.Scatter(
            x=[lo, hi], y=[y, y],
            mode="lines",
            line=dict(color=color, width=4),
            opacity=0.45,
            showlegend=False,
            hoverinfo="skip",
        ))

    # mean marker
    fig.add_trace(go.Scatter(
        x=mmms, y=y_positions,
        mode="markers",
        name=scen_label,
        marker=dict(color=color, size=11, line=dict(color="white", width=1)),
        hovertemplate=(
            f"<b>{scen_label}</b><br>"
            "MMM: %{x:.2f}°C<br>"
            "Range: %{customdata[0]:.2f}–%{customdata[1]:.2f}°C"
            "<extra></extra>"
        ),
        customdata=list(zip(los, his)),
    ))

all_los = [results[h][s]["lo"] for h in horizons for s in scen_list]
all_his = [results[h][s]["hi"] for h in horizons for s in scen_list]
pad = 0.1
x_range = [min(all_los) - pad, max(all_his) + pad]

fig.update_layout(
    title=dict(
        text=(
            "Period-mean California temperature anomaly by scenario<br>"
            "<sup>Dots = multi-model mean · Lines = full model spread (±1 STD across GCMs)</sup>"
        ),
        font=dict(size=14),
    ),
    xaxis=dict(
        title="Temperature anomaly (°C) relative to 1981–2010",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
        rangemode="tozero",
        range=x_range,
    ),
    yaxis=dict(
        title=None,
        tickmode="array",
        tickvals=list(range(len(horizons))),
        ticktext=horizons,
        showgrid=False,
        showline=False,
        zeroline=False,
        autorange="reversed",  # near-term at top, end-of-century at bottom
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        x=0.80, y=0.80,
        xanchor="right", yanchor="bottom",
        font=dict(size=11),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#dddddd",
        borderwidth=0.5,
    ),
    width=800, height=480,
    margin=dict(l=140, r=30, t=90, b=60),
)

fig.show(config=config)
fig.write_html(
    "../uncertainty/figures/html/scenario_period_bars.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("../uncertainty/figures/static/scenario_period_bars.png", scale=2)

## Step 3: Scenario vs. model uncertainty over time

The plume chart shows this qualitatively; here we quantify it formally. We
estimate two variance components at each year:

- **Scenario uncertainty** $S(t)$: variance of the three scenario MMMs
  around their grand mean — *how much does the answer change depending on
  which emissions pathway you assume?*
- **Model uncertainty** $M(t)$: average within-scenario variance across
  models — *how much do the 8 models disagree, given the same scenario?*

Their sum approximates total uncertainty (excluding internal variability, which
requires large ensembles — see `internal_variability_guidance.ipynb`).

The **crossover point** — where $S(t)$ exceeds $M(t)$ — is the year after
which scenario choice matters more than model choice. This is the key
quantitative finding.

In [ ]:
# Align all three scenario timeseries to the same years
years = ts_245.time.dt.year.values

# Stack scenario MMMs into a (3, n_years) array
mmm_245 = ts_245.mean(dim="sim").values   # (n_years,)
mmm_370 = ts_370.mean(dim="sim").values
mmm_585 = ts_585.mean(dim="sim").values

scenario_mmms = np.stack([mmm_245, mmm_370, mmm_585], axis=0)  # (3, n_years)

# ── Scenario uncertainty: variance across the 3 MMMs ─────────────────────
# ddof=1 for an unbiased estimate with only 3 scenarios
S = np.var(scenario_mmms, axis=0, ddof=1)   # (n_years,)

# ── Model uncertainty: mean within-scenario variance ─────────────────────
# For each scenario, variance across its 8 models at each year; then average
var_245 = ts_245.var(dim="sim").values           # (n_years,)
var_370 = ts_370.var(dim="sim").values
var_585 = ts_585.var(dim="sim").values

M = np.mean(np.stack([var_245, var_370, var_585], axis=0), axis=0)  # (n_years,)

# Total
T = S + M

# 10-year rolling smooth on all components
def smooth(arr, w=10):
    return pd.Series(arr, index=years).rolling(w, center=True, min_periods=5).mean().values

S_s, M_s, T_s = smooth(S), smooth(M), smooth(T)

# Fractional contributions
frac_S = np.where(T_s > 0, S_s / T_s, np.nan)
frac_M = np.where(T_s > 0, M_s / T_s, np.nan)

# Crossover year: first year where scenario variance exceeds model variance
crossover_mask = S_s > M_s
crossover_year = int(years[crossover_mask][0]) if crossover_mask.any() else None
print(f"Scenario uncertainty exceeds model uncertainty after: {crossover_year}")

In [ ]:
fig = go.Figure()

# ── Left panel traces (xaxis="x", yaxis="y") ─────────────────────────────
fig.add_trace(go.Scatter(
    x=years, y=S_s,
    name="Scenario uncertainty",
    line=dict(color="#C0664A", width=2.2),
    xaxis="x", yaxis="y",
    hovertemplate="Year: %{x}<br>Scenario variance: %{y:.3f}°C²<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=years, y=M_s,
    name="Model uncertainty",
    line=dict(color="#4472A8", width=2.2),
    xaxis="x", yaxis="y",
    hovertemplate="Year: %{x}<br>Model variance: %{y:.3f}°C²<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=years, y=T_s,
    name="Total",
    line=dict(color="black", width=2.0, dash="dot"),
    opacity=0.6,
    xaxis="x", yaxis="y",
    hovertemplate="Year: %{x}<br>Total variance: %{y:.3f}°C²<extra></extra>",
))

# ── Right panel traces (xaxis="x2", yaxis="y2") ───────────────────────────
# Scenario fraction fill (bottom, orange)
fig.add_trace(go.Scatter(
    x=np.concatenate([years, years[::-1]]),
    y=np.concatenate([frac_S, np.zeros(len(years))]),
    fill="toself",
    fillcolor="rgba(192,102,74,0.75)",
    line=dict(width=0),
    name="Scenario fraction",
    xaxis="x2", yaxis="y2",
    hoverinfo="skip",
    showlegend=True,
))

# Model fraction fill (top, blue)
fig.add_trace(go.Scatter(
    x=np.concatenate([years, years[::-1]]),
    y=np.concatenate([np.ones(len(years)), frac_S[::-1]]),
    fill="toself",
    fillcolor="rgba(68,114,168,0.75)",
    line=dict(width=0),
    name="Model fraction",
    xaxis="x2", yaxis="y2",
    hoverinfo="skip",
    showlegend=True,
))

# Invisible hover line on right panel
fig.add_trace(go.Scatter(
    x=years, y=frac_S,
    mode="lines",
    line=dict(width=0),
    showlegend=False,
    xaxis="x2", yaxis="y2",
    hovertemplate=(
        "Year: %{x}<br>"
        "Scenario: %{y:.1%}<br>"
        "Model: %{customdata:.1%}"
        "<extra></extra>"
    ),
    customdata=frac_M,
))

# ── Crossover vertical lines ───────────────────────────────────────────────
if crossover_year:
    # Left panel crossover line
    fig.add_trace(go.Scatter(
        x=[crossover_year, crossover_year],
        y=[0, max(T_s[~np.isnan(T_s)])],
        mode="lines",
        line=dict(color="#888888", width=1.2, dash="dash"),
        xaxis="x", yaxis="y",
        showlegend=False,
        hoverinfo="skip",
    ))
    # Right panel crossover line
    fig.add_trace(go.Scatter(
        x=[crossover_year, crossover_year],
        y=[0, 1],
        mode="lines",
        line=dict(color="lightgrey", width=1.2, dash="dash"),
        xaxis="x2", yaxis="y2",
        showlegend=False,
        hoverinfo="skip",
    ))
    # Crossover annotation on left panel
    fig.add_annotation(
        x=crossover_year + 1,
        y=max(T_s[~np.isnan(T_s)]) * 0.95,
        xref="x", yref="y",
        text=f"Crossover<br>~{crossover_year}",
        showarrow=False,
        font=dict(size=9, color="#666666"),
        xanchor="left", yanchor="top",
    )

# 50% reference line on right panel
fig.add_trace(go.Scatter(
    x=[years[0], years[-1]],
    y=[0.5, 0.5],
    mode="lines",
    line=dict(color="lightgrey", width=1, dash="dash"),
    xaxis="x2", yaxis="y2",
    showlegend=False,
    hoverinfo="skip",
))

# Subplot titles as annotations
fig.add_annotation(
    text="Uncertainty components — California temperature",
    x=0.225, y=1.06, xref="paper", yref="paper",
    showarrow=False, font=dict(size=12),
    xanchor="center",
)
fig.add_annotation(
    text="Fractional uncertainty — scenario vs. model",
    x=0.775, y=1.06, xref="paper", yref="paper",
    showarrow=False, font=dict(size=12),
    xanchor="center",
)

# ── Layout ────────────────────────────────────────────────────────────────
fig.update_layout(
    width=1000, height=460,
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    xaxis=dict(
        domain=[0, 0.45],
        matches="x2",              # ← links hover to right panel
        title_text="Year",
        showgrid=True, gridcolor="#eeeeee",
        zeroline=False, showline=False,
    ),
    xaxis2=dict(
        domain=[0.55, 1.0],
        matches="x",               # ← links hover to left panel
        title_text="Year",
        showgrid=True, gridcolor="#eeeeee",
        zeroline=False, showline=False,
        anchor="y2",
    ),
    yaxis=dict(
        domain=[0, 1],
        title_text="Variance (°C²)",
        showgrid=True, gridcolor="#eeeeee",
        zeroline=False, showline=False,
    ),
    yaxis2=dict(
        domain=[0, 1],
        title_text="Fraction of total variance",
        tickformat=".0%",
        range=[0, 1],
        showgrid=True, gridcolor="#eeeeee",
        zeroline=False, showline=False,
        anchor="x2",
    ),
    legend=dict(
        x=0.02, y=0.97,
        xanchor="left", yanchor="top",
        font=dict(size=10),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#dddddd",
        borderwidth=0.5,
    ),
    margin=dict(l=60, r=30, t=90, b=60),
)

fig.show(config=config)
fig.write_html(
    "../uncertainty/figures/html/scenario_vs_model_variance.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("../uncertainty/figures/static/scenario_vs_model_variance.png", scale=2)

**Reading the panels:**

Left panel (absolute variance): both components grow over time, but scenario
uncertainty grows faster and crosses model uncertainty around the printed
crossover year. Before the crossover, running multiple models matters more than
choosing between scenarios. After it, the reverse is true.

Right panel (fractional contributions): the orange fill shows the scenario
fraction of total variance; the blue fill shows the model fraction. The
dashed white line at 0.5 marks equal contributions. Early in the century the
chart is mostly blue (model-dominated); late in the century it is mostly orange
(scenario-dominated).

> **Practical implication for ensemble design:** If your application is
> end-of-century and the crossover has already occurred, including all three
> scenarios is more important than maximizing the number of models per scenario.
> If your application is near-term (< 2040), scenario choice barely affects the
> result — focus instead on model spread and internal variability.

## Step 4: Spatial scenario sensitivity maps

The timeseries above is a California-wide average. Scenario uncertainty is not
spatially uniform — some regions are more sensitive to emissions pathway than
others. The maps below show this directly.

We compare the **multi-model mean temperature change** under SSP2-4.5 and
SSP5-8.5 over 2071–2100, then plot their **difference** (the spatial field of
scenario uncertainty). A third panel shows the same difference for
precipitation, illustrating how differently the two variables respond.

### Step 4a: Retrieve and compute spatial fields

In [ ]:
# End-of-century period: 2071–2100
eoc_slice = slice("2071", "2100")

# Historical mean per simulation (spatial field, for anomaly calculation)
# hist_mean_per_sim already computed in Step 1c

def eoc_spatial_anom(future_ds, hist_mean):
    """
    Return end-of-century mean temperature anomaly as a spatial field,
    averaged across models (multi-model mean).
    Shape: (y, x)
    """
    anom = future_ds["tasmax"] - hist_mean          # anomaly vs. 1981–2010
    eoc  = anom.sel(time=eoc_slice).mean(dim="time")
    mmm  = eoc.mean(dim="sim")                   # average across models
    return mmm.compute()


print("Computing spatial fields...")
tas_eoc_245 = eoc_spatial_anom(ssp245_future, hist_mean_per_sim)
tas_eoc_585 = eoc_spatial_anom(ssp585_future, hist_mean_per_sim)
tas_eoc_diff = tas_eoc_585 - tas_eoc_245   # scenario difference map

print(f"Temperature spatial fields computed. Spatial shape: {tas_eoc_245.shape}")

### Step 4b: Three-panel comparison maps

Each panel uses the same colorbar limits so the maps are directly comparable.
The **difference map** (right panel) is the spatial field of scenario
uncertainty — where it is darkest, the choice of emissions pathway matters most
for that grid cell.

In [ ]:
def make_tas_map_cartopy(data, title, vmin, vmax, diverging=False,
                          ax=None, extent=[-125.75, -111.25, 31, 44.75]):
    """Temperature anomaly map with cartopy projection and coastlines."""
    # cmap_name = "ae_diverging_r" if diverging else "ae_orange"
    # cmap_hex  = read_ae_colormap(cmap=cmap_name, cmap_hex=True)
    # cmap      = LinearSegmentedColormap.from_list(cmap_name, cmap_hex)
    if diverging:
        cmap_name = "ae_diverging"
        cmap_hex  = read_ae_colormap(cmap=cmap_name, cmap_hex=True)
        cmap      = LinearSegmentedColormap.from_list(cmap_name, cmap_hex)
    else:
        cmap = plt.cm.viridis

    display_crs = ccrs.PlateCarree()
    standalone  = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5, 6), subplot_kw={"projection": display_crs})

    vals = np.array(data.values, dtype=float)
    
    if diverging:
        vmin = np.floor(np.nanmin(vals)*10)/10
        vmin = max(vmin,0)
        vmax = np.round(np.nanmax(vals)*10)/10
        levels = np.arange(vmin, vmax+0.001, 0.1)
        if vmin>=0:
            cmap_name = "ae_orange"
            cmap_hex  = read_ae_colormap(cmap=cmap_name, cmap_hex=True)
            cmap      = LinearSegmentedColormap.from_list(cmap_name, cmap_hex)
    else:
        levels = np.arange(vmin, vmax+0.0001, 0.5)
        
    im = ax.contourf(
        data["lon"].values, data["lat"].values, vals,
        levels=levels,
        cmap=cmap, vmin=vmin, vmax=vmax,
        transform=display_crs,
    )
    # Overlay a pcolormesh for the colorbar (contourf colorbar formatting is limited)
    # im = ax.pcolormesh(
    #     data["lon"].values, data["lat"].values, vals,
    #     cmap=cmap, vmin=vmin, vmax=vmax,
    #     transform=display_crs, alpha=0, shading="auto",
    # )

    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="k")
    ax.add_feature(cfeature.STATES,    linewidth=0.3, edgecolor="#555555")
    ax.add_feature(cfeature.OCEAN,     facecolor="#e8f4f8", zorder=0)
    ax.set_extent(extent, crs=display_crs)
    ax.set_title(title, fontsize=10, pad=6)
    plt.colorbar(im, ax=ax, orientation="vertical",
                 label="°C", shrink=0.85, pad=0.02)

    # if standalone:
    #     plt.tight_layout()
    #     plt.show()
    return ax

In [ ]:
# ── Colour scales ─────────────────────────────────────────────────────────
def to_plotly_colorscale(hex_list):
    n = len(hex_list)
    return [[i / (n - 1), c] for i, c in enumerate(hex_list)]

cmap_hex_warm   = read_ae_colormap(cmap="ae_orange", cmap_hex=True)
cmap_hex_warm   = read_ae_colormap(cmap="ae_diverging", cmap_hex=True)
colorscale_diff = to_plotly_colorscale(cmap_hex_warm)
colorscale_warm = "viridis"

norm_diff = tas_eoc_diff
norm_diff = (tas_eoc_diff-tas_eoc_diff.mean())/tas_eoc_diff.mean()

# ── Shared limits ─────────────────────────────────────────────────────────
all_vals = np.concatenate([
    tas_eoc_245.values.flatten(),
    tas_eoc_585.values.flatten(),
])
vmax_abs = float(np.nanpercentile(all_vals, 98))
min_diff = float(np.floor(np.nanmin(norm_diff) * 10) / 10)
diff_abs = float(np.nanpercentile(np.abs(norm_diff.values), 98))
min_diff = -0.6
diff_abs = 0.6

# ── Map panel helper ──────────────────────────────────────────────────────
def add_map_panel(fig, da, row, col, colorscale, zmin, zmax, colorbar_title,
                  n_levels=11):
    vals   = da.values.astype(float)
    lon_1d = da["lon"].values
    lat_1d = da["lat"].values
    if lon_1d.ndim > 1:
        lon_1d = lon_1d[0, :]
        lat_1d = lat_1d[:, 0]

    fig.add_trace(
        go.Contour(
            z=vals, x=lon_1d, y=lat_1d,
            colorscale=colorscale,
            zmin=zmin, zmax=zmax,
            contours=dict(
                start=zmin, end=zmax,
                size=(zmax - zmin) / (n_levels - 1),
                coloring="fill", showlines=False,
            ),
            showscale=False,
            hovertemplate=(
                "Lon: %{x:.2f}°<br>Lat: %{y:.2f}°<br>"
                f"{colorbar_title}: %{{z:.2f}}°C<extra></extra>"
            ),
            name=colorbar_title, showlegend=False,
        ),
        row=row, col=col,
    )

    # Coastlines + state borders
    for feature, color, width in [
        (cfeature.NaturalEarthFeature("physical", "coastline", "10m"),
         "black", 0.8),
        (cfeature.NaturalEarthFeature("cultural",
         "admin_1_states_provinces_lines", "10m"),
         "#777777", 0.5),
    ]:
        for geom in feature.geometries():
            for line in getattr(geom, "geoms", [geom]):
                coords = np.array(line.coords)
                lons_b, lats_b = coords[:, 0], coords[:, 1]
                if np.max(lons_b) < -126 or np.min(lons_b) > -111:
                    continue
                if np.max(lats_b) < 31 or np.min(lats_b) > 45:
                    continue
                fig.add_trace(
                    go.Scatter(
                        x=lons_b, y=lats_b, mode="lines",
                        line=dict(color=color, width=width),
                        showlegend=False, hoverinfo="skip",
                        xaxis=f"x{col if col > 1 else ''}",
                        yaxis=f"y{col if col > 1 else ''}",
                    ),
                    row=row, col=col,
                )

# ── Build figure ──────────────────────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "SSP2-4.5 MMM (2071–2100)",
        "SSP5-8.5 MMM (2071–2100)",
        "Scenario difference<br>SSP5-8.5 − SSP2-4.5<br>Fractional deviation from spatial mean",
    ],
    specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "xy"}]],
    horizontal_spacing=0.05,
)

add_map_panel(fig, tas_eoc_245,  1, 1, colorscale_warm, 0,        vmax_abs, "°C")
add_map_panel(fig, tas_eoc_585,  1, 2, colorscale_warm, 0,        vmax_abs, "°C")
add_map_panel(fig, norm_diff, 1, 3, colorscale_diff, min_diff, diff_abs, "Δ°C")

# ── Shared colorbars via go.Heatmap dummy traces ──────────────────────────
# go.Heatmap aligns horizontal colorbar ticks correctly unlike go.Contour

# Colorbar 1 — shared for left + center panels
n_ticks  = 6
cb1_vals = np.linspace(0, vmax_abs, n_ticks)

fig.add_trace(go.Heatmap(
    z=np.linspace(0, vmax_abs, 100).reshape(10, 10),
    colorscale=colorscale_warm,
    zmin=0, zmax=vmax_abs,
    showscale=True,
    colorbar=dict(
        title=dict(text="°C", font=dict(size=11), side="top"),
        orientation="h",
        x=0.30, xanchor="center",
        y=-0.06, yanchor="top",
        len=0.58, thickness=16,
        tickfont=dict(size=10),
        tickmode="array",
        tickvals=cb1_vals.tolist(),
        ticktext=[f"{v:.1f}" for v in cb1_vals],
    ),
    showlegend=False,
    xaxis="x", yaxis="y",
    opacity=0,
    visible=True,
))

# Colorbar 2 — right panel
n_ticks  = 6
cb2_vals = np.linspace(min_diff, diff_abs, n_ticks)

fig.add_trace(go.Heatmap(
    z=np.linspace(min_diff, diff_abs, 100).reshape(10, 10),
    colorscale=colorscale_diff,
    zmin=min_diff, zmax=diff_abs,
    showscale=True,
    colorbar=dict(
        title=dict(text="Δ°C", font=dict(size=11), side="top"),
        orientation="h",
        x=0.835, xanchor="center",
        y=-0.06, yanchor="top",
        len=0.28, thickness=16,
        tickfont=dict(size=10),
        tickmode="array",
        tickvals=cb2_vals.tolist(),
        ticktext=[f"{v:.1f}" for v in cb2_vals],
    ),
    showlegend=False,
    xaxis="x", yaxis="y",
    opacity=0,
    visible=True,
))

# ── Axes extent ───────────────────────────────────────────────────────────
lon_min, lon_max = -124.5, -111.5
lat_min, lat_max =   32.0,   43.5

for col in [1, 2, 3]:
    xkey = f"xaxis{col if col > 1 else ''}"
    ykey = f"yaxis{col if col > 1 else ''}"
    yref = f"y{col if col > 1 else ''}"
    fig.update_layout(**{
        xkey: dict(
            range=[lon_min, lon_max],
            showticklabels=False,
            showgrid=False, zeroline=False, showline=False,
            scaleanchor=yref, scaleratio=1,
        ),
        ykey: dict(
            range=[lat_min, lat_max],
            showticklabels=False,
            showgrid=False, zeroline=False, showline=False,
        ),
    })

# ── Push subplot titles up away from maps ────────────────────────────────
for ann in fig.layout.annotations:
    ann.y   = ann.y + 0.06
    ann.font = dict(size=12)

# ── Final layout ──────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="California temperature anomaly at end-of-century — scenario sensitivity",
        font=dict(size=14), x=0.5, xanchor="center", y=0.99,
    ),
    width=900, height=488,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=10, r=10, t=100, b=110),
)

fig.show(config=config)
fig.write_html(
    "../uncertainty/figures/html/scenario_spatial_maps.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("../uncertainty/figures/static/scenario_spatial_maps.png", scale=2)

**Reading the maps:** The left and centre panels show the multi-model mean
temperature change under the two bookend scenarios. Both show broadly similar
spatial patterns — more warming inland than on the coast, amplified warming at
elevation — but with the SSP5-8.5 values uniformly larger.

The right panel is the most informative: the **difference map is the spatial
field of scenario uncertainty**. Where it is darkest, the choice of emissions
pathway matters most for that location's projected warming. The interior valleys
and Sierra Nevada typically show larger values than the coast, reflecting
greater amplification of the GHG-forced signal in continental, high-altitude
environments.

## Step 5: When does scenario choice matter?

The analyses in Steps 2–4 consistently show the same pattern. The guidance
below synthesizes it into actionable rules.

| Context | Scenario uncertainty dominates? | Recommendation |
|---|---|---|
| Near-term (< 2040), any variable | **No** — all SSPs converge | Use any single scenario; note that near-term results are scenario-insensitive |
| Mid-century (2041–2070), temperature | **Partial** — crossing over | Include at least two SSPs (2-4.5 and 5-8.5) to bracket the range |
| End-of-century (2071–2100), temperature | **Yes** — dominant source | Include all three SSPs; scenario spread exceeds model spread |
| End-of-century, precipitation | **Partial** — internal variability still large | Include all SSPs but also note high within-scenario uncertainty |
| Planning for extremes | **Yes** — must represent high end | Always include SSP5-8.5; never use SSP2-4.5 alone for risk |
| Mitigation benefit communication | **Yes** — the whole point | Compare SSP2-4.5 and SSP5-8.5 directly; the difference map is the message |

**Practical checklist for any scenario uncertainty analysis:**
1. **What is your planning horizon?** If < 2040, scenario choice barely
   affects your result — note this explicitly to your audience and focus
   on model spread and internal variability instead.
2. **What is your variable?** Temperature has a cleaner scenario signal than
   precipitation. For precipitation, the uncertainty decomposition from
   `uncertainty_decomposition.ipynb` will show whether internal variability
   or scenario uncertainty dominates your specific question.
3. **Are you communicating risk or central tendency?** For risk applications
   (infrastructure design, emergency planning), always include the high
   scenario (SSP5-8.5). Using only the middle scenario (SSP3-7.0) or the
   MMM systematically underestimates the upper tail of possible outcomes.
4. **Are you showing spatial results?** Generate a difference map
   (SSP5-8.5 − SSP2-4.5) as a companion to any single-scenario map. It
   immediately reveals where your spatial conclusions are sensitive to
   scenario choice and where they are robust.
5. **Are multiple scenarios impractical?** If you can only include one, use
   SSP3-7.0 as the central estimate — but document the choice and report
   the spread from Step 2c so readers can assess its significance.

> For a quantitative decomposition of how scenario uncertainty compares to
> model uncertainty and internal variability over time, see
> `uncertainty_decomposition.ipynb`.

**References:**
- [IPCC AR6 Synthesis Report, Section 3](https://www.ipcc.ch/report/ar6/syr/):
  scenario dependence of future climate outcomes
- [Tebaldi et al. (2024) — npj Climate Action](https://www.nature.com/articles/s44168-023-00082-1):
  lessons from AR6 on integrating scenarios across working groups
- [IPCC uncertainty guidance note](https://www.ipcc.ch/site/assets/uploads/2018/05/uncertainty-guidance-note.pdf):
  formal framework for communicating scenario uncertainty
- [ClimateWest: Uncertainty 101](https://climatewest.ca/2022/09/27/uncertainty-101-understanding-climate-models/):
  accessible treatment of all three uncertainty sources including scenario
- [PARC uncertainty primer](https://data.parc.ca/uncertainty-primer/):
  regional adaptation planning context for scenario uncertainty